In [1]:
!pip install jams

In [2]:
import argparse
import json
import random
from pathlib import Path
from typing import Dict, List, Tuple
import jams
import numpy as np
import soundfile as sf
from tqdm import tqdm

In [3]:
# -----------------------------------------------------------------------------
# Configuration ----------------------------------------------------------------
# -----------------------------------------------------------------------------

# Root directory that contains the GuitarSet subset (adjust if relocating)
DATASET_ROOT = Path('/kaggle/input/guitarset')
ANNOTATION_DIR = DATASET_ROOT / "annotation"
AUDIO_DIR = DATASET_ROOT / "audio_mono-mic"
OUTPUT_ROOT = Path('/kaggle/working/') / "chords"

# Train/Val/Test split ratios (only used when --split is enabled)
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Random seed for reproducibility
RANDOM_SEED = 42

# All 12 chromatic roots
CHROMATIC_ROOTS = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

# Target chords → output folder name mapping
# Each key is a (root, quality) pair as used in Harte notation
# We include all 12 roots for both major and minor chords (24 classes total)
TARGET_CHORDS: Dict[Tuple[str, str], str] = {}
for root in CHROMATIC_ROOTS:
    TARGET_CHORDS[(root, "maj")] = root           # e.g., ("C", "maj") -> "C"
    TARGET_CHORDS[(root, "min")] = f"{root}m"     # e.g., ("C", "min") -> "Cm"

# -----------------------------------------------------------------------------
# Augmentation settings for under-represented classes --------------------------
# -----------------------------------------------------------------------------

# Minimum number of examples for a class to be considered "well-represented"
MIN_EXAMPLES_THRESHOLD = 90

# Window duration for extracting snippets (in seconds)
WINDOW_DURATION = 1.0

# Minimum duration required to split an audio into multiple windows
# (must be at least WINDOW_DURATION to extract one full segment)
MIN_DURATION_FOR_SPLIT = WINDOW_DURATION

In [4]:
# -----------------------------------------------------------------------------
# Helpers ----------------------------------------------------------------------
# -----------------------------------------------------------------------------

def _ensure_dirs(use_split: bool) -> None:
    """Create output folders for each target chord (with or without splits)."""
    if use_split:
        for split in ["train", "val", "test"]:
            for folder in TARGET_CHORDS.values():
                (OUTPUT_ROOT / split / folder).mkdir(parents=True, exist_ok=True)
    else:
        for folder in TARGET_CHORDS.values():
            (OUTPUT_ROOT / folder).mkdir(parents=True, exist_ok=True)


def _canonicalize(label: str) -> Tuple[str, str]:
    """Return (root, quality) ignoring extensions/inversions.

    Example: "C:maj/3" -> ("C", "maj")
    """
    if label == "N":
        return ("N", "N")

    # Remove inversion (slash chords)
    label = label.split("/")[0]
    try:
        root, extra = label.split(":", 1)
    except ValueError:
        # malformed, treat as non-target
        return ("", "")

    # Remove extensions e.g. "maj7(9)" → "maj7"
    extra = extra.split("(")[0]

    # Strip numerals (maj7 -> maj, min9 -> min)
    extra = extra.rstrip("0123456789")
    return root, extra


def _match_target(label: str):
    """Return True if *label* matches a target chord without extensions.

    Rules:
    • Inversions ("/3") are ignored.
    • Any numeral in the chord quality (e.g., "maj7", "min9", "7") indicates an extension → reject.
    """
    # Remove inversion first to inspect the quality string
    core = label.split("/")[0]
    if ":" not in core:
        return False

    root, qual = core.split(":", 1)
    # Reject if quality contains any digit (extension) before optional parentheses
    if any(ch.isdigit() for ch in qual.split("(")[0]):
        return False

    return _canonicalize(label) in TARGET_CHORDS


def _split_data(items: List, train_ratio: float, val_ratio: float) -> Tuple[List, List, List]:
    """Split a list into train/val/test sets."""
    n = len(items)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    
    train = items[:n_train]
    val = items[n_train:n_train + n_val]
    test = items[n_train + n_val:]
    
    return train, val, test


def _compute_windows(duration: float, window_size: float) -> List[Tuple[float, float]]:
    """Compute non-overlapping windows of window_size seconds from a segment.
    
    Returns a list of (start_offset, end_offset) tuples relative to the segment start.
    If duration < window_size, returns empty list (snippet will be kept as-is elsewhere).
    """
    windows = []
    num_full_windows = int(duration // window_size)
    
    for i in range(num_full_windows):
        start_offset = i * window_size
        end_offset = start_offset + window_size
        windows.append((start_offset, end_offset))
    
    return windows


def _expand_snippets_for_underrepresented(
    snippets_by_chord: Dict[str, List[dict]],
    threshold: int,
    window_duration: float
) -> Dict[str, List[dict]]:
    """Expand under-represented classes by splitting longer clips into multiple windows.
    
    For classes with fewer than `threshold` examples:
    - Clips >= 2*window_duration are split into multiple non-overlapping windows
    - Clips >= window_duration but < 2*window_duration are kept as one window
    - Clips < window_duration are kept as-is (will be padded during training)
    
    Returns a new dict with expanded snippets.
    """
    expanded = {}
    
    for chord_name, snippets in snippets_by_chord.items():
        original_count = len(snippets)
        
        if original_count >= threshold:
            # Well-represented class: keep original snippets
            expanded[chord_name] = snippets.copy()
            continue
        
        # Under-represented class: expand by splitting longer clips
        new_snippets = []
        
        for snippet in snippets:
            duration = snippet["duration"]
            
            if duration >= 2 * window_duration:
                # Can split into multiple windows
                windows = _compute_windows(duration, window_duration)
                for i, (start_off, end_off) in enumerate(windows):
                    new_snippets.append({
                        "wav_path": snippet["wav_path"],
                        "start": snippet["start"] + start_off,
                        "end": snippet["start"] + end_off,
                        "duration": window_duration,
                        "out_name": snippet["out_name"].replace(".wav", f"_w{i}.wav"),
                        "is_window": True,
                    })
            else:
                # Keep as single snippet (either full clip or will be padded)
                new_snippets.append(snippet)
        
        expanded[chord_name] = new_snippets
        
        # Log expansion
        if len(new_snippets) > original_count:
            print(f"  {chord_name}: {original_count} -> {len(new_snippets)} examples (+{len(new_snippets) - original_count})")
    
    return expanded

In [5]:
# -----------------------------------------------------------------------------
# Core extraction loop ---------------------------------------------------------
# -----------------------------------------------------------------------------

def extract_snippets(use_split: bool = True, augment_underrepresented: bool = True):
    """Extract chord snippets from GuitarSet.
    
    Args:
        use_split: If True, split into train/val/test (80/10/10).
                   If False, put all snippets in one folder per chord class.
        augment_underrepresented: If True, expand under-represented classes by 
                                   splitting longer audio clips into multiple 2s windows.
    """
    _ensure_dirs(use_split)
    
    # Set random seed for reproducibility
    random.seed(RANDOM_SEED)

    # First pass: collect all snippet metadata grouped by chord class
    # Structure: {chord_folder: [(wav_path, start, end, duration, out_name), ...]}
    snippets_by_chord: Dict[str, List[dict]] = {name: [] for name in TARGET_CHORDS.values()}

    jam_files: List[Path] = sorted(ANNOTATION_DIR.glob("*.jams"))
    print("Phase 1: Scanning annotations and collecting snippet metadata...")
    
    for jam_path in tqdm(jam_files, desc="Scanning JAMS (comp only)"):
        stem = jam_path.stem  # e.g. 00_BN1-129-Eb_comp

        # Skip solo performances
        if not stem.endswith("_comp"):
            continue

        wav_path = AUDIO_DIR / f"{stem}_mic.wav"
        if not wav_path.exists():
            # Some files may be missing mono mic version
            continue

        jam = jams.load(str(jam_path))
        chord_ann = jam.search(namespace="chord")
        if not chord_ann:
            continue
        # First annotator
        chord_ann = chord_ann[0]
        data = chord_ann.data

        for event in data:
            label = event.value
            if not _match_target(label):
                continue
            root, quality = _canonicalize(label)
            chord_folder = TARGET_CHORDS[(root, quality)]

            start = event.time
            end = event.time + event.duration
            out_name = f"{stem}_{start:.3f}_{end:.3f}.wav".replace("/", "-")
            
            # Store metadata for later extraction
            snippets_by_chord[chord_folder].append({
                "wav_path": wav_path,
                "start": start,
                "end": end,
                "duration": event.duration,
                "out_name": out_name,
            })

    # Print original class distribution
    print("\nOriginal class distribution:")
    for chord_name in sorted(snippets_by_chord.keys()):
        count = len(snippets_by_chord[chord_name])
        marker = " *" if count < MIN_EXAMPLES_THRESHOLD else ""
        print(f"  {chord_name:>4}: {count:>3} examples{marker}")
    
    # Augment under-represented classes if enabled
    if augment_underrepresented:
        print(f"\nPhase 1.5: Augmenting under-represented classes (threshold={MIN_EXAMPLES_THRESHOLD})...")
        snippets_by_chord = _expand_snippets_for_underrepresented(
            snippets_by_chord, 
            MIN_EXAMPLES_THRESHOLD, 
            WINDOW_DURATION
        )
        
        # Print new distribution
        print("\nNew class distribution after augmentation:")
        for chord_name in sorted(snippets_by_chord.keys()):
            count = len(snippets_by_chord[chord_name])
            print(f"  {chord_name:>4}: {count:>3} examples")

    # Second pass: shuffle (if splitting), split (if enabled), and extract audio
    if use_split:
        print("\nPhase 2: Shuffling, splitting (80/10/10), and extracting audio...")
        summary = {
            "train": {name: {"count": 0, "seconds": 0.0} for name in TARGET_CHORDS.values()},
            "val": {name: {"count": 0, "seconds": 0.0} for name in TARGET_CHORDS.values()},
            "test": {name: {"count": 0, "seconds": 0.0} for name in TARGET_CHORDS.values()},
        }
    else:
        print("\nPhase 2: Extracting audio (no split)...")
        summary = {name: {"count": 0, "seconds": 0.0} for name in TARGET_CHORDS.values()}
    
    # Cache loaded audio files to avoid reloading
    audio_cache: Dict[Path, Tuple[np.ndarray, int]] = {}
    
    for chord_folder, snippets in tqdm(snippets_by_chord.items(), desc="Processing chords"):
        if not snippets:
            continue
        
        if use_split:
            # Shuffle snippets for this chord class
            random.shuffle(snippets)
            
            # Split into train/val/test
            train_snippets, val_snippets, test_snippets = _split_data(
                snippets, TRAIN_RATIO, VAL_RATIO
            )
            
            # Process each split
            for split_name, split_snippets in [("train", train_snippets), 
                                                ("val", val_snippets), 
                                                ("test", test_snippets)]:
                for snippet_info in split_snippets:
                    wav_path = snippet_info["wav_path"]
                    
                    # Load audio (with caching)
                    if wav_path not in audio_cache:
                        y, sr = sf.read(str(wav_path))
                        audio_cache[wav_path] = (y, sr)
                    else:
                        y, sr = audio_cache[wav_path]
                    
                    start = snippet_info["start"]
                    end = snippet_info["end"]
                    start_frame = int(np.floor(start * sr))
                    end_frame = int(np.ceil(end * sr))
                    snippet = y[start_frame:end_frame]
                    
                    if snippet.size == 0:
                        continue
                    
                    out_path = OUTPUT_ROOT / split_name / chord_folder / snippet_info["out_name"]
                    sf.write(str(out_path), snippet, sr)
                    
                    # Update summary
                    summary[split_name][chord_folder]["count"] += 1
                    summary[split_name][chord_folder]["seconds"] += snippet_info["duration"]
        else:
            # No split - just extract all snippets
            for snippet_info in snippets:
                wav_path = snippet_info["wav_path"]
                
                # Load audio (with caching)
                if wav_path not in audio_cache:
                    y, sr = sf.read(str(wav_path))
                    audio_cache[wav_path] = (y, sr)
                else:
                    y, sr = audio_cache[wav_path]
                
                start = snippet_info["start"]
                end = snippet_info["end"]
                start_frame = int(np.floor(start * sr))
                end_frame = int(np.ceil(end * sr))
                snippet = y[start_frame:end_frame]
                
                if snippet.size == 0:
                    continue
                
                out_path = OUTPUT_ROOT / chord_folder / snippet_info["out_name"]
                sf.write(str(out_path), snippet, sr)
                
                # Update summary
                summary[chord_folder]["count"] += 1
                summary[chord_folder]["seconds"] += snippet_info["duration"]

    # Write summary JSON
    summary_path = OUTPUT_ROOT / "snippet_summary.json"
    with summary_path.open("w", encoding="utf-8") as fp:
        json.dump(summary, fp, indent=2)

    # Print summary statistics
    print("\n" + "=" * 60)
    print("EXTRACTION COMPLETE")
    print("=" * 60)
    
    if use_split:
        for split_name in ["train", "val", "test"]:
            total_count = sum(s["count"] for s in summary[split_name].values())
            total_seconds = sum(s["seconds"] for s in summary[split_name].values())
            print(f"{split_name.upper():>6}: {total_count:>5} snippets, {total_seconds:.1f} seconds")
    else:
        total_count = sum(s["count"] for s in summary.values())
        total_seconds = sum(s["seconds"] for s in summary.values())
        print(f"TOTAL: {total_count} snippets, {total_seconds:.1f} seconds")
    
    print("=" * 60)
    print(f"Summary saved to: {summary_path}")


In [6]:
# Set augment_underrepresented=True to expand minor chord classes by splitting longer clips
# Set augment_underrepresented=False to keep original behavior
extract_snippets(use_split=True, augment_underrepresented=True)

Phase 1: Scanning annotations and collecting snippet metadata...


Scanning JAMS (comp only): 100%|██████████| 360/360 [02:07<00:00,  2.82it/s]



Original class distribution:
     A:  96 examples
    A#: 102 examples
   A#m:  48 examples *
    Am:  36 examples *
     B:  96 examples
    Bm:  36 examples *
     C: 114 examples
    C#: 114 examples
   C#m:  48 examples *
    Cm:  24 examples *
     D: 102 examples
    D#: 108 examples
   D#m:  24 examples *
    Dm:  36 examples *
     E: 108 examples
    Em:  60 examples *
     F: 144 examples
    F#:  96 examples
   F#m:  24 examples *
    Fm:  36 examples *
     G: 126 examples
    G#: 114 examples
   G#m:  48 examples *
    Gm:  60 examples *

Phase 1.5: Augmenting under-represented classes (threshold=90)...
  Cm: 24 -> 36 examples (+12)
  C#m: 48 -> 72 examples (+24)
  Dm: 36 -> 84 examples (+48)
  D#m: 24 -> 72 examples (+48)
  Em: 60 -> 84 examples (+24)
  Fm: 36 -> 144 examples (+108)
  F#m: 24 -> 48 examples (+24)
  Gm: 60 -> 120 examples (+60)
  G#m: 48 -> 108 examples (+60)
  Am: 36 -> 48 examples (+12)
  A#m: 48 -> 120 examples (+72)
  Bm: 36 -> 72 examples (+36)

New 

Processing chords: 100%|██████████| 24/24 [00:22<00:00,  1.05it/s]


EXTRACTION COMPLETE
 TRAIN:  1852 snippets, 3682.7 seconds
   VAL:   222 snippets, 442.1 seconds
  TEST:   254 snippets, 489.9 seconds
Summary saved to: /kaggle/working/chords/snippet_summary.json
